# Chapter 3 — Embedding
### Mapping Word2Vec Vector Semantics

This notebook trains Word2Vec embeddings (Skip-gram) on BPE tokens to study the semantic associations of key character vectors.

In [ ]:
from esther_nlp.data import load_chapters_and_verses
from esther_nlp.tokenizer import EstherBpeTokenizer
from esther_nlp.embeddings import train_word2vec
from esther_nlp.analysis import get_radial_ranks
from esther_nlp.plots import fig_03_radial_map, fig_06b_embedding_ball
from pathlib import Path
from IPython.display import Image, display
import numpy as np
from sklearn.decomposition import PCA

_, all_verses = load_chapters_and_verses()
tokenizer = EstherBpeTokenizer(vocab_size=1000)
tokenizer.train(all_verses)
tokenized_verses = [tokenizer.encode_to_tokens(v) for v in all_verses]

# Train Word2Vec Skip-gram model with 10 seeds
SEEDS = [42, 1, 7, 13, 21, 99, 100, 314, 1729, 2025]
models = [train_word2vec(tokenized_verses, sg=1, epochs=100, seed=s, vector_size=50) for s in SEEDS]
base_model = models[0]

# Compute character radial rankings from המלך
sorted_chars = ["המן", "המלכה", "אסתר", "מרדכי", "ושתי", "אחשורוש"]
vocab_size = len(base_model.wv.key_to_index)
per_char = {c: [] for c in sorted_chars}

for m in models:
    ranks = get_radial_ranks(m, "המלך", sorted_chars)
    for c in sorted_chars:
        r, _ = ranks.get(c, (-1, 0.0))
        per_char[c].append(r)

medians = {c: int(np.median(per_char[c])) for c in sorted_chars}
sorted_by_med = sorted(sorted_chars, key=lambda c: medians[c])

# Plot Radial target Map
fig_radial_path = Path("fig_03_radial_map.png")
fig_03_radial_map(sorted_by_med, medians, per_char, vocab_size, fig_radial_path)
display(Image(filename=fig_radial_path))

# Plot 3D Vector Ball Projection
ball_chars = ["המלך", "המן", "מרדכי", "אסתר", "אחשורוש"]
ball_vectors = np.stack([base_model.wv[c] for c in ball_chars])
vn = ball_vectors / (np.linalg.norm(ball_vectors, axis=1, keepdims=True) + 1e-12)
p_pca = PCA(n_components=3).fit(vn)
pos_3d = p_pca.transform(vn)
pos_3d = pos_3d / (np.linalg.norm(pos_3d, axis=1, keepdims=True) + 1e-12)

fig_ball_path = Path("fig_06b_embedding_ball.png")
fig_06b_embedding_ball(pos_3d, ball_chars, fig_ball_path)
display(Image(filename=fig_ball_path))

## Vector Embeddings Analysis & Conclusion

### Core Observations
1. **Haman sits closest to the Throne**: In our Skip-gram semantic coordinates space, Haman sits at **rank 22** out of all 640 vocabulary words relative to `המלך` (within the top 3% of closeness). He is mathematically closer to the concept of the throne than any other protagonist.
2. **Achashverosh sits furthest**: Paradoxically, the personal name of the human king, `אחשורוש`, sits at **rank 218** — the furthest distance of all main characters.

### Narrative and Literary Conclusion
This geometric separation outlines the central tragedy of the royal court in Shushan. The personal name `אחשורוש` appears only 29 times, mostly in the early chapters, before being completely swallowed by the formal title `המלך` from chapter 4 onward. The man is completely consumed by the state. Haman's entire plot revolves around usurping the institutional power of the crown, and the vector map registers this: Haman resides in the immediate lexical context of the institution (`המלך`), whereas the human king is displaced to a distant semantic sector. The geometry maps the office, not the person.